In [1]:
from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader=PyPDFLoader("Cashless Card Solutions - Educational Sector (3).pdf")
pdf_doc = loader.load()

In [4]:
with open("pdf_doc_complete.txt", "w", encoding="utf-8") as f:
    for doc in pdf_doc:
        f.write(doc.page_content + "\n\n")

In [5]:
from langchain_community.document_loaders import RecursiveUrlLoader
from bs4 import BeautifulSoup

# Create a loader with max_depth=3
loader = RecursiveUrlLoader(
    url="https://synthetechcloud.com",
    max_depth=2,  # Crawl up to 3 levels deep
    extractor=lambda x: BeautifulSoup(x, "html.parser").text,
    prevent_outside=True  # Stay within the original domain
)

documents = loader.load()

In [6]:
import re

def clean_text(text):
    # Remove excessive whitespace and newlines
    text = re.sub(r'\s+', ' ', text)
    # Remove non-printable characters
    text = ''.join(c for c in text if c.isprintable())
    return text.strip()

cleaned_documents = []
for doc in documents:
    cleaned_content = clean_text(doc.page_content)
    # Create a new Document with cleaned content and original metadata
    cleaned_doc = type(doc)(page_content=cleaned_content, metadata=doc.metadata)
    cleaned_documents.append(cleaned_doc)

cleaned_documents

[Document(metadata={'source': 'https://synthetechcloud.com', 'content_type': 'text/html', 'title': 'SYNTHETECH CLOUD SOLUTIONS - HOME', 'description': 'Discover the future of streamlined operations and industry excellence with Synthetech Cloud Solutions. Our cutting-edge ERP solutions empower healthcare, education, and retail sectors, revolutionizing patient care, enhancing learning experiences, and reshaping retail landscapes. Seamlessly integrate data, automate tasks, and make informed decisions. Explore the power of tailored technology solutions that elevate efficiency, engagement, and success across industries.', 'language': 'en'}, page_content='SYNTHETECH CLOUD SOLUTIONS - HOME HOME ABOUT INDUSTRIES Healthcare Clinic Management Hospital Management LAB Management Pharmacy POS Cashless Campus Solution Education School Management College Management Library Management Hostel Management Cashless Campus Solutions Retail Ecommerce Point of Sale Supply Chain Finance Loyalty and Rewards Pr

In [12]:
from langchain_openai import ChatOpenAI
from tqdm import tqdm

# Initialize the OpenAI 4o mini model
llm = ChatOpenAI(model="gpt-4o-mini")

def process_for_rag(raw_content):
    prompt = (
        "You are a helpful assistant. Clean and summarize the following raw text for use in a Retrieval-Augmented Generation (RAG) system. "
        "Remove irrelevant information, boilerplate, and make the content concise, factual, and ready for semantic search.\n\n"
        f"Raw content:\n{raw_content}\n\nProcessed content:"
    )
    return llm.invoke(prompt).content.strip()
with open("cleaned_documents.txt", "w", encoding="utf-8") as f:
    for doc in tqdm(cleaned_documents, desc="Processing documents"):
        processed_content = process_for_rag(doc.page_content)
        f.write(processed_content + "\n\n")


Processing documents: 100%|██████████| 52/52 [04:53<00:00,  5.64s/it]


In [13]:
from langchain_community.document_loaders import TextLoader

txt_loader1 = TextLoader("D:/Projest/sales_chatbot/cleaned_documents.txt")
txt_loader2 = TextLoader("D:/Projest/sales_chatbot/pdf_doc_complete.txt")

txt_docs1 = txt_loader1.load()
txt_docs2 = txt_loader2.load()

all_txt_docs = txt_docs1 + txt_docs2

In [14]:
all_txt_docs

[Document(metadata={'source': 'D:/Projest/sales_chatbot/cleaned_documents.txt'}, page_content='**Synthetech Cloud Solutions Overview**\n\nSynthetech Cloud Solutions provides tailored cashless solutions across various industries including healthcare, education, retail, banking, and hospitality. Their offerings include:\n\n- **Products:**\n  - Cashless Card Solutions\n  - Ecommerce and Loyalty Management Systems\n  - Wallet Management Systems\n  - Core Banking Solutions\n  - Point of Sale Systems for Quick Service Restaurants (QSR)\n  - School and Hospital Management Systems\n  - Custom Application Development (Web and Mobile)\n\n- **Key Benefits:**\n  - **Seamless Integration:** Easy transition from cash-based to cashless systems.\n  - **Enhanced Efficiency:** Streamline operations and improve customer satisfaction.\n  - **Security and Scalability:** Secure systems designed to grow with your business.\n  - **Expert Support:** Dedicated guidance from setup through ongoing success.\n\nSyn

In [87]:
len(doc)

64

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=250,
    length_function=len
)
texts = text_splitter.split_documents(all_txt_docs)

len(texts)

67

In [17]:
import os 
import dotenv
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
model="text-embedding-3-small"


from langchain_openai import OpenAIEmbeddings , ChatOpenAI

embeddings = OpenAIEmbeddings(
  
    model=model
)

In [18]:
from langchain_community.vectorstores import FAISS

# Initialize the FAISS vector store
vector_store = FAISS.from_documents(
    documents=texts,
    embedding=embeddings,
)

# Save the FAISS vector store to local storage
vector_store.save_local("vector_store")

# Load the FAISS vector store from local storage
vector_store = FAISS.load_local("vector_store", embeddings,allow_dangerous_deserialization=True)

In [92]:
vector_store

In [19]:
retriver=vector_store.as_retriever()
retriver    

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000162FD5BD4C0>, search_kwargs={})

In [20]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
)

In [21]:
llm.invoke("hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0392822090', 'id': 'chatcmpl-BX0nUri1In7RRJrPoawUbQkmFqDnd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ff1d14bc-f7f8-4635-b48a-f9b258fe4779-0', usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [22]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Define a custom prompt template
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert in providing information about cashless card solutions. Use the following context to answer the user's question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:"""
)

# Create the RAG QA chain with the custom prompt
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,  # Replace with your LLM, e.g., OpenAI(model="gpt-3.5-turbo")
    chain_type="stuff",
    retriever=retriver,
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

qa_chain

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an expert in providing information about cashless card solutions. Use the following context to answer the user's question.\nIf the answer is not in the context, say you don't know.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000162FD36ED20>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000162FD5BE450>, root_client=<openai.OpenAI object at 0x00000162FD876DB0>, root_async_client=<openai.AsyncOpenAI object at 0x00000162FD36FF50>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********')), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input

In [24]:
query = "how the smartcard is helpfull  ?"
result = qa_chain({"query": query})
print(result["result"])

The smartcard is helpful in several ways:

1. **Integrated Wallet**: It has a built-in wallet for all campus-related activities, allowing students to make payments, access restricted areas, and perform other functions all in one card.

2. **Contactless Transactions**: Students can conduct transactions at every touchpoint without the need for cash, which speeds up processes and reduces the need for frequent visits to cash counters.

3. **Faster Processing**: The card facilitates quick offline transactions, processed rapidly, improving the overall user experience.

4. **Real-Time Balance**: Users can see their real-time balance on counter slips for every transaction, enhancing transparency and financial management.

5. **Convenience**: Features like instant card loading facilities on campus and automated report generation make it easier for students, staff, and vendors to manage their transactions.

6. **Enhanced Security**: The optional personalized PIN feature provides an additional la